[![My Notebooks](https://github.com/KelianB/SPARK/raw/main/assets/teaser.webp)](https://github.com/yhu9/MonoFaceCompute)

# How to Run the SPARK Model to Generate Your Own 3D Facial Reconstruction
---

This is based off the cutting edge facial reconstruction model SPARK!. Link to the model
repository [here](https://github.com/KelianB/MonoFaceCompute/tree/main).

## Steps in this Tutorial

In this tutorial, we are going to cover:

- **Before you start** — configure your HuggingFace / Roboflow API keys, confirm GPU access, and set up the `HOME` directory.
- **Install required packages** — download the sample Obama videos from HuggingFace and clone + set up the `MonoFaceCompute` repository.
- **Video Dataset Preprocessing** — write a dataset `.yaml` config and run
  `MonoFaceCompute` to extract, crop, matte, segment, find landmarks, track,
  and optimize flame model to each video frame.

## 🔥 Let's begin!


# Before you start
### Configure your API key
- Open your HuggingFace Settings page. Click Access Tokens then New Token to generate new token.
- Go to your Roboflow Settings page. Click Copy. This will place your private key in the clipboard.
- In Colab, go to the left pane and click on Secrets (🔑).
    - Store HuggingFace Access Token under the name HF_TOKEN.
    - Store Roboflow API Key under the name ROBOFLOW_API_KEY

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

!which conda mamba

In [ ]:
!pip install -U huggingface_hub

import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## Select the runtime
Let's make sure that we have access to a GPU. We can use the `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

**NOTE:** To make it easier for us to manage datasets, images and models we create a `HOME` constant.

In [ ]:
import os
HOME = os.getcwd()
print("HOME:", HOME)

## Install required packages

There are two main repos to run SPARK, MonoFaceCompute and SPARK. For
convenience, I uploaded the precomputation of obama videos on hugging face so
you don't have to go through that process! However, if you want to perform the
precomputation on your own videos so you can run SPARK on your own videos,
follow the next steps and then try it out on your own dataset.

In [ ]:
%cd {HOME}
from huggingface_hub import snapshot_download

local_dir = snapshot_download(
    repo_id="masawho/obama-face-videos",
    repo_type="dataset",
    local_dir=os.path.join(HOME, "obama-face-videos")
)
print("Downloaded dataset to:", local_dir)

!git clone https://github.com/yhu9/MonoFaceCompute.git

%cd {HOME}/MonoFaceCompute
!bash setup.sh

This may take a while to complete. Please make sure to wait until the setup.sh
completes fully.

## Run Precomputation of sample obama videos or your own video data

To run MonoFaceCompute on your own video dataset, you'll need to adjust
example.txt for your own dataset and computer directory. Open
MonoFaceCompute/datasets/example.yaml and adjust base_dir and output_dir to
your desired fields. For convenience, you can always use the example obama
video and the following generates the obama

In [ ]:
import os
from pathlib import Path

p = Path(os.getcwd())
home = p.parent                       # e.g. /home/<user>
base_dir = home / "obama-face-videos" / "obama"
output_dir = home / "obama-face-videos" / "obama_processed"

out_path = Path("datasets/obama.yaml")
out_path.parent.mkdir(parents=True, exist_ok=True)   # ensure the datasets/ dir exists

yaml_content = f"""\
base_dir: {base_dir}
output_dir: {output_dir}
tracker: EMOCA
shape_tracker: SMIRK
crop_mode: constant
crop_scale: 1.4
resize: 512
sequences:
    1: {{ source: 1.mp4 }}
    2: {{ source: 2.mp4 }}
    3: {{ source: 3.mp4 }}
    4: {{ source: 4.mp4 }}
    6: {{ source: 6.mp4 }}
shape_sequence: 1
steps: ["extract", "crop", "matte", "segment", "landmarks", "track", "optimize"]
"""

out_path.write_text(yaml_content)
print(f"Wrote {out_path.resolve()}")
print(yaml_content)
print(os.path.exists(out_path.resolve()))

In [ ]:
!python process.py --dataset datasets/obama.yaml

## 🏆 Congratulations

You've reached the end of this notebook! 

### Learning Resources

If you wish to run SPARK on the processed video, follow the tutorial found here.

- [How to use SPARK](https://github.com/roboflow/notebooks):